# 연구 문제 정의와 분석 스프린트

## 공개자료를 실행 가능한 연구문제로 전환하기

개인 또는 조별 관심사를 관측 단위, target, baseline, split, metric과 검증 범위가 명확한 최소 분석으로 구체화합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 현실의 문제를 한 문장의 설명·예측 질문으로 바꿉니다.
- grain, target, feature, baseline, split과 metric을 포함한 분석 명세를 작성합니다.
- Bike 실행경로와 개인·조별 분석계획 경로를 구분하고, 각 경로에 맞는 결과 또는 계획을 기록합니다.
- 12교시가 이어서 읽을 `lesson11_project_record.json`에 경로·상태·근거를 저장합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: Steel 최소 예측문제

35,040개의 15분 Steel 관측에서 다음 시점 `Usage_kWh`를 예측하는 문제를 예시로 사용합니다. grain, target, forecast origin, exact lag, persistence baseline, chronological split과 MAE만 남겨 최소 분석을 구성합니다.

### 적용 실습: 공통 Bike Sharing 자료 또는 개인·조별 자료

공통 실습은 17,379개의 Bike 시간별 자료를 사용합니다. 자신의 자료를 사용할 경우에는 코드를 실행하기 전에 다음 정보를 먼저 제시해야 합니다.

| 분석 전 확인 항목 | 기록할 내용 |
|---|---|
| 출처와 사용권 | 제공기관, URL 또는 파일 provenance, 이용조건 |
| 관측 단위 | 한 행이 나타내는 대상·공간·시간 |
| 표본 범위 | 행 수, 기간, 집단과 제외 기준 |
| 변수 역할 | target, 사용 가능한 feature, 식별자, 누수 후보 |
| 품질 | 결측, 중복 key, 단위, 범주 code |

개인·조별 원자료를 ChatGPT에 업로드하지 않습니다. schema, 익명화한 요약, 실행 결과와 정확한 오류 메시지만 사용합니다.

Practice는 두 경로 가운데 하나를 `artifacts/lesson11_project_record.json`에 저장합니다.

| 경로 | 저장 상태 | 이 교시에서 완성할 것 |
|---|---|---|
| `bike` | `analysis_result` | 기준모형·후보 결과, 한계, 60초 결과 공유문 |
| `personal_canvas` | `analysis_plan` | 승인 전 Problem Canvas, 사용권·정보시점·다음 실행; 성능수치 없음 |

12교시 Practice는 이 파일을 읽으므로 저장 경로와 `track`·`status`를 마지막 출력에서 확인합니다.


## 연구 관심사에서 분석 질문으로

**Python 분석 초점:** 모형 이름보다 grain·target·baseline·split·metric을 먼저 표로 만들고, 실행 가능한 최소 분석으로 연결합니다.

`에너지 사용을 줄이고 싶다`는 중요한 연구 관심사이지만, 이 문장만으로는 분석할 관측, 결과변수, 비교조건, 평가기준을 정할 수 없습니다.

다음 질문은 자료와 평가절차를 구체적으로 지정합니다.

> 2018년 15분 운전자료에서 과거 사용량과 운전조건으로 다음 15분 `Usage_kWh`를 예측할 때, 후보모형의 미래 test MAE가 persistence보다 낮은가?

분석 질문에는 대상과 기간, 한 행의 단위, 결과변수, 비교대상, 평가척도가 드러나야 합니다.


In [ ]:
import pandas as pd

canvas = pd.DataFrame({
    "항목": ["현실 문제", "한 행(grain)", "target", "예측간격", "baseline", "split", "metric"],
    "예시 기록": [
        "다음 시간 자전거 수요를 비교한다", "한 도시의 한 시간", "다음 한 시간 cnt",
        "1시간", "직전 시간 수요", "과거 train → 미래 test", "MAE (건/시간)",
    ],
    "코드에서 연결할 객체": [
        "problem_text", "timestamp 한 행", "y", "lag_1h", "baseline", "train/test", "mean_absolute_error",
    ],
})
print("[현실 문제를 Python 분석 객체로 번역]")
display(canvas)
print("모형 이름은 표의 빈칸을 채운 뒤에 선택합니다.")


## 연구질문의 네 유형

| 질문 유형 | 예 | 필요한 근거 |
|---|---|---|
| 기술 | 사용량이 언제 높았는가 | 집계, 분포, 시각화 |
| 연관성 | 어떤 변수와 배출량이 함께 변하는가 | 회귀명세, 불확실성, 진단 |
| 예측 | 보지 않은 미래 값을 얼마나 잘 예측하는가 | 독립적인 split, baseline, metric |
| 인과 | 개입이 결과를 얼마나 변화시켰는가 | 반사실을 구성하는 식별전략 |

질문의 유형이 달라지면 필요한 자료, 비교설계, 적절한 결과 표현도 달라집니다.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

question_types = pd.DataFrame({
    "질문 유형": ["기술", "비교", "예측", "분해"],
    "plot_label": ["Describe", "Compare", "Predict", "Decompose"],
    "Python 산출물": ["요약표·분포", "그룹 차이·회귀", "baseline·미관측 오차", "요인별 기여도"],
    "이 수업의 예": ["CO₂ 분포", "STIRPAT 연관성", "전력·수요 예측", "LMDI 기여도"],
    "인과효과 자동 확보": [False, False, False, False],
})
print("[질문 유형에 따라 달라지는 산출물]")
display(question_types.drop(columns="plot_label"))

output_matrix = np.eye(4, dtype=int)
fig, ax = plt.subplots(figsize=(8.5, 3.5))
sns.heatmap(
    output_matrix, annot=np.where(output_matrix == 1, "use", ""), fmt="",
    cbar=False, cmap="Blues", linewidths=0.5,
    yticklabels=question_types["plot_label"],
    xticklabels=["Summary /\ndistribution", "Group difference /\nregression", "Baseline /\nunseen error", "Factor\ncontribution"],
    ax=ax,
)
ax.set(xlabel="Python output", ylabel="Question type", title="The question determines the required output")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


## 연구문제 명세표

본 과정에서는 분석을 시작하기 전에 다음 여덟 항목을 한 표에 기록합니다. 이 표를 수업용 Problem Canvas로 사용합니다.

| 항목 | 기록할 내용 |
|---|---|
| 현실 문제 | 누가 어떤 결정을 내려야 하는가 |
| 분석 질문 | 자료로 답할 수 있는 한 문장 |
| 관측 단위 | 한 행이 나타내는 대상·공간·시간 |
| 결과변수 | 예측하거나 설명할 값 또는 사건 |
| 비교기준 | 후보방법과 비교할 단순 기준 |
| 분할·비교 | 학습·평가 또는 집단 비교 방식 |
| 평가척도 | 성공을 판단할 수치 또는 추정량 |
| 결과 사용자 | 결과를 사용할 사람과 의사결정 |

모형은 이 명세가 정리된 뒤에 선택합니다.


## 제한시간 안에 완결하는 최소 실행 분석

이 수업에서 최소 실행 분석은 질문부터 결과와 한계까지 한 번 연결되는 가장 작은 분석을 뜻합니다. 특정 학술방법의 명칭이 아니라 연구설계를 조기에 점검하기 위한 작업 원칙입니다.

`자료 읽기 → grain/key → 결과·입력변수 → split → baseline → 후보 1개 → metric → 표 또는 그림 1개 → 한계`

이 경로를 먼저 완결하면 자료가 질문에 답할 수 있는지, 결과정보 누수가 없는지, 공정한 비교가 가능한지를 초기에 확인할 수 있습니다. 이후의 변수와 모형 확장은 이 기본 분석을 기준으로 판단합니다.


## 네 단계 연구 검토

| 검토 단계 | 확인할 질문 |
|---|---|
| 문제 명세 | 질문 유형, 관측 단위, 결과변수가 분명한가 |
| 데이터 사용 | 출처, 사용권, 표본범위, 누수 후보를 확인했는가 |
| 실행 재현성 | 새 kernel에서 같은 절차와 결과가 재현되는가 |
| 결과 채택 | baseline, split, metric, 한계가 기록됐는가 |

이 네 단계는 Pineau et al. (2021)의 reproducibility checklist를 수업의 분석과정에 맞게 재구성한 검토 절차입니다. 논문의 체크리스트 자체와 동일한 네 단계라고 해석하지 않습니다.

출처: [Pineau et al. (2021)](https://www.jmlr.org/papers/v22/20-303.html)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

gate = pd.DataFrame({
    "gate": ["문제", "데이터", "실행", "결과"],
    "plot_label": ["Problem", "Data", "Execution", "Result"],
    "확인": ["grain·target·결정", "출처·사용권·누수", "split·baseline·metric", "표·그림·주장·한계"],
    "plot_text": ["grain / target / decision", "source / license / leakage", "split / baseline / metric", "table / figure / claim / limit"],
    "예시 통과": [True, True, False, False],
})
print("[앞 단계가 실패하면 뒤 단계의 결론을 보류]")
display(gate.drop(columns=["plot_label", "plot_text"]))

fig, ax = plt.subplots(figsize=(8, 2.3))
colors = ["#54A24B" if ok else "#E45756" for ok in gate["예시 통과"]]
ax.barh(gate["plot_label"], [1] * len(gate), color=colors)
for i, text in enumerate(gate["plot_text"]):
    ax.text(0.02, i, text, va="center", color="white", fontweight="bold")
ax.set(xlim=(0, 1), xticks=[], title="Research gates: stop at the first failure")
plt.tight_layout(); plt.show()
print("이 예시는 실행 gate에서 멈춥니다. 뒤의 결과를 먼저 만들지 않습니다.")


## 생성형 AI를 이용한 연구설계 검토

### 제공할 수 있는 정보

- 공개자료의 열 이름과 익명화된 스키마
- 실행한 코드의 관련 부분과 정확한 오류 메시지
- 공개 가능한 요약통계와 결과표

### 제공하지 않는 정보

- 개인정보, 미공개 원자료, 기관 기밀
- 비밀번호, API key, 개인 식별정보

요청 범위는 `누수 가능 열 검토`, `split의 타당성 점검`, `결과문장과 표의 일치 여부 확인`처럼 검증 가능한 단위로 좁힙니다. 제안된 열 이름, 단위, split, metric은 실제 Notebook과 대조하고 채택 여부와 근거를 기록합니다.


## 60초 연구설계 공유

| 순서 | 공통 내용 | `bike / analysis_result` | `personal_canvas / analysis_plan` |
|---:|---|---|---|
| 1 | 현실 문제와 분석 질문 | 결과 질문 | 계획 질문 |
| 2 | 한 행과 결과변수 | 평가기간 | 예정 표본·정보시점 |
| 3 | baseline·split·metric | 실제 비교조건 | 계획한 비교조건 |
| 4 | 현재 확보한 근거 | 대표 결과수치 | 아직 결과수치가 없음을 명시 |
| 5 | 한계와 다음 검증 | 결과의 적용범위 | 사용권 확인과 다음 실행 |

두 경로 모두 `artifacts/lesson11_project_record.json`에 저장합니다. 동료 검토는 모형 추가보다 관측 단위, 결과정보 누수, split, baseline, 주장 범위를 먼저 확인하며, 계획경로에 임의의 성능수치나 결과주장을 만들지 않습니다.


## 실행 환경과 입력자료

강의 시연은 Steel 최소 예측문제이고, Practice는 두 작업경로 중 하나를 선택합니다.

| 경로 | 확인할 항목 | 11교시 산출물 |
|---|---|---|
| `bike` | grain·target·정보시점·baseline·split·metric | `analysis_result`와 결과표·그림 |
| `personal_canvas` | 출처·사용권·grain·target·비교설계·다음 실행 | `analysis_plan`; 모형·성능수치 없음 |

두 경로 모두 `artifacts/lesson11_project_record.json`을 저장하며, 12교시 Practice가 이 파일을 그대로 읽습니다.

> **실행 전 확인:** 지금 선택한 경로는 결과를 만드는 경로입니까, 실행 전 분석계획을 승인받는 경로입니까?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 연구 관심사를 실행 가능한 질문으로 구체화

> 관심사: 에너지 사용을 줄이고 싶다.

> 분석 질문: 과거 15분 사용량과 운전조건으로 다음 15분 `Usage_kWh`를 예측할 때, 후보모형의 미래 test MAE가 persistence보다 낮은가?

| 명세 요소 | Steel 시연의 기준 |
|---|---|
| 관측 단위 | 15분 운전 관측 |
| 결과변수 | 다음 시점 `Usage_kWh` |
| 예측 기준시점 | 현재 15분 시점 |
| 비교기준 | 직전값 persistence |
| 평가설계 | 과거→미래 분할, MAE |

질문과 명세가 일치해야 필요한 데이터, 입력변수, 비교절차를 결정할 수 있습니다.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

analysis_name = "Steel 최소 분석 시연"
canvas = pd.DataFrame({
    "항목": ["현실 문제", "분석 질문", "관측 단위", "결과변수", "비교 기준모형", "자료 분할", "평가척도", "결과 사용자"],
    "기록": ["설비 전력수요를 미리 파악", "최근 수요와 달력정보로 다음 15분 사용량을 예측할 수 있는가?",
             "한 설비의 15분", "Usage_kWh", "직전 15분 사용량", "과거 80% 학습 → 미래 20% 평가",
             "MAE (kWh)", "설비 운영·부하계획 담당자"],
})
display(canvas)


### 출력 해석: 현실 문제에서 분석 명세까지

문제·데이터·방법·평가·의사결정이 끊기지 않고 연결됐는지 한 줄씩 읽어봅니다.


In [ ]:
canvas_view = canvas.copy()
canvas_view.index = range(1, len(canvas_view) + 1)
display(canvas_view)
print("질문 → 관측 단위·결과변수 → 기준모형·분할·평가척도 → 결과 사용자 순서로 연결합니다.")


## 2. 제한시간 안에 완결하는 최소 실행 분석

`자료 읽기 → grain/key → exact lag → 시간순 split → baseline → 후보 1개 → MAE → 예측표`

각 단계가 연구질문의 다음 단계로 이어져야 합니다. 모형을 적합했더라도 같은 미래기간의 baseline 비교가 없다면 후보방법의 추가가치를 평가할 수 없습니다.

### 완료 기준

- baseline과 후보의 결과표 1개
- 동일한 test 행의 첫 예측값
- 평가기간과 단위가 포함된 결과문장 1개
- 현재 설계에서 확인하지 못한 한계 1개


In [ ]:
raw = pd.read_csv(find_course_data("steel/Steel_industry_data.csv"))
raw["timestamp"] = pd.to_datetime(raw["date"], dayfirst=True)
raw = raw.sort_values("timestamp")
lag = raw[["timestamp", "Usage_kWh"]].copy(); lag["timestamp"] += pd.Timedelta(minutes=15)
data = raw.merge(lag.rename(columns={"Usage_kWh":"lag_15m"}), on="timestamp", how="left").dropna(subset=["lag_15m"]).copy()
data["hour"] = data["timestamp"].dt.hour
features = ["lag_15m", "hour", "NSM"]
cut = int(len(data)*.8); train, test = data.iloc[:cut], data.iloc[cut:]
model = HistGradientBoostingRegressor(max_iter=100, random_state=42).fit(train[features], train["Usage_kWh"])
prediction = model.predict(test[features]); baseline = test["lag_15m"]
result_table = pd.DataFrame([
    {"model":"persistence", "MAE":mean_absolute_error(test["Usage_kWh"], baseline)},
    {"model":"hist gradient boosting", "MAE":mean_absolute_error(test["Usage_kWh"], prediction)},
])
display(result_table)
result_by_model = result_table.set_index("model")
baseline_mae = float(result_by_model.loc["persistence", "MAE"])
candidate_mae = float(result_by_model.loc["hist gradient boosting", "MAE"])
mae_change = baseline_mae - candidate_mae
mae_direction = "낮았습니다" if mae_change >= 0 else "높았습니다"
evaluation_period = f"{test['timestamp'].min():%Y-%m-%d %H:%M}–{test['timestamp'].max():%Y-%m-%d %H:%M}"
result_claim = (
    f"{evaluation_period}의 미래 평가자료에서 후보모형 MAE는 {candidate_mae:.2f} kWh로, "
    f"직전 값 기준모형의 {baseline_mae:.2f} kWh보다 {abs(mae_change):.2f} kWh {mae_direction}."
)
limitation = "단일 설비·기간·분할의 예측 비교이며 다른 계절·설비와 정책 인과효과는 검증하지 않았습니다."
print(result_claim)
print("한계:", limitation)


### 출력 해석: 최소 결과의 구성

동일한 미래 test에서 baseline과 후보를 비교하고 첫 예측 행을 확인합니다.


In [ ]:
print("train end:", train["timestamp"].max())
print("test start:", test["timestamp"].min())
minimum_result_view = pd.DataFrame({
    "timestamp": test["timestamp"].iloc[:8].to_numpy(),
    "observed": test["Usage_kWh"].iloc[:8].to_numpy(),
    "baseline": baseline.iloc[:8].to_numpy(),
    "candidate": prediction[:8],
})
display(minimum_result_view)


## 3. 사전 중단 기준

| 검토 항목 | 계속 진행할 조건 | 수정이 필요한 조건 |
|---|---|---|
| 데이터 | 관측 단위·key·단위가 분명함 | 결과변수 또는 key가 불명확함 |
| 정보시점 | 모든 입력이 예측시점에 가용함 | 미래정보나 결과정보가 포함됨 |
| 비교설계 | baseline·split·metric이 있음 | 후보모형의 점수만 제시됨 |
| 실행 | 새 kernel에서 재현됨 | 숨은 변수나 경로에 의존함 |

분석 전에 중단 기준을 정하면 결과를 본 뒤 표본이나 모형을 자의적으로 바꾸는 일을 줄일 수 있습니다.


In [ ]:
steel_usage_by_time = raw.drop_duplicates("timestamp").set_index("timestamp")["Usage_kWh"]
expected_lag_15m = (data["timestamp"] - pd.Timedelta(minutes=15)).map(steel_usage_by_time)
validation_gate = pd.Series({
    "target not in features": "Usage_kWh" not in features,
    "exact lag source exists": expected_lag_15m.notna().all(),
    "lag values match source timestamp": np.allclose(data["lag_15m"], expected_lag_15m),
    "strict time order": train["timestamp"].max() < test["timestamp"].min(),
    "baseline present": "persistence" in set(result_table["model"]),
    "metric unit recorded": canvas.loc[canvas["항목"].eq("평가척도"), "기록"].astype(str).str.contains("kWh", case=False, regex=False).all(),
}, name="passed")
display(validation_gate.to_frame())


## 4. 분석 완료 기준

- 연구질문과 데이터의 관측 단위가 연결됩니다.
- baseline과 후보를 같은 평가표본에서 비교합니다.
- 표 또는 그림이 핵심 결과를 직접 제시합니다.
- 결과문장에 metric, 기간, 비교대상이 포함됩니다.
- 가장 직접적인 한계와 다음 검증을 기록합니다.

후보모형이 baseline보다 나쁜 경우도 공정한 비교가 이루어졌다면 분석결과에 포함합니다. 이 결과는 이후 변수, 모형, 자료를 확장할 근거가 됩니다.


In [ ]:
checks = validation_gate.copy(); checks["finite result"] = np.isfinite(result_table["MAE"]).all()
checks["claim evidence complete"] = evaluation_period in result_claim and all(f"{v:.2f}" in result_claim for v in [candidate_mae, baseline_mae]) and "kWh" in result_claim
checks["limitation recorded"] = len(limitation) >= 30
display(checks.to_frame()); assert checks.all()
print("연구 분석흐름 검증: PASS")


## ChatGPT 실시간 검토: 제한시간 안의 연구설계 다듬기

연구질문과 자료 요약을 제공하고, 독립 실습에 배정된 약 22분 안에 완결할 수 있는 최소 분석인지 동료검토를 요청합니다. 답변은 많은 분석 아이디어가 아니라 **가장 큰 설계 위험 하나와 다음 행동 하나**를 우선해야 합니다.

개인·조별 원자료는 업로드하지 않고 공개 스키마, 익명화한 요약, 실행 결과와 정확한 오류 메시지만 사용합니다.


In [ ]:
review_prompt = f"""
다음 공개자료 분석을 22분 안에 완결할 수 있는 최소 연구로 검토해 주세요.

- 분석: {analysis_name}
- Problem Canvas:\n{canvas.to_string(index=False)}
- 실제 입력변수: {features}
- 예측간격: 15분
- 학습 종료시각: {train['timestamp'].max()}
- 평가 시작시각: {test['timestamp'].min()}
- 현재 결과표:\n{result_table.to_string(index=False)}

아래 형식으로 답해 주세요.
1. 현재 연구질문과 Problem Canvas의 일치 여부: 통과 / 수정 필요 / 정보 부족
2. 한 행·예측대상·입력정보 시점·기준모형·분할·평가지표
3. 가장 큰 설계 위험 1개와 확인 방법
4. 지금 추가할 작업 1개 / 이번 시간에 버릴 작업 1개
5. 제가 바로 수행할 다음 행동 1개

제약: 제공되지 않은 정보는 추정하지 말고 ‘정보 부족’으로 표시하세요. 분석을 대신 완성하거나 많은 모델을 나열하지 말고, 제공되지 않은 열과 결과를 만들지 마세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### 실시간 ChatGPT 답변을 받은 뒤

1. 제안이 연구문제 명세의 어느 칸을 보완하는지 표시합니다.
2. 남은 실습시간 안에 실행할 변경 한 가지만 선택하고 나머지는 후속과제로 남깁니다.
3. 채택한 제안과 제외한 제안의 근거를 각각 한 문장으로 기록합니다.


## 해석 범위

공통 Steel 예제는 제한시간 안에 연구질문과 최소 분석을 완결하는 연습 사례입니다. 수강생의 실제 연구문제와 자료를 대신하지 않습니다.

현재 결과는 선택한 데이터·기간·모형에 한정됩니다. 실제 과제로 확장할 때에는 자료 사용권, 도메인 가정, 운영비용, 외부 타당성과 필요한 윤리 검토를 연구설계에 추가합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

연구질문을 한 행·예측대상·기준모형·분할·평가지표로 번역하고, 제한시간 안에 검증 가능한 최소 결과와 가장 큰 설계 위험을 남깁니다.

### 적용 순서

1. `11_research_sprint_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `11_research_sprint_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 현실 문제를 한 문장의 분석 질문으로 바꾸면 무엇입니까?
- 한 행, target, baseline, split, metric을 모두 명시했습니까?
- 주어진 실습시간 안에 끝까지 검증할 수 있는 가장 작은 분석은 무엇입니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
